# Hugging FaceのLLMをGGUFへ変換し、README付きで再公開するNotebook

## このNotebookの位置づけ

このNotebookは、**Hugging Face上のLLMを `GGUF` 形式へ変換し、必要に応じて量子化し、README付きでHugging Faceへ再公開する**ためのものです。  
学習そのものを行うNotebookではなく、**学習済みモデルの配布・ローカル推論・検証をしやすくするための後処理Notebook**です。

このNotebookで扱う代表的な流れは次の通りです。

1. 変換元モデルを取得する  
   - Hugging Face上の通常のTransformers形式モデル
   - ローカルにある `.gguf`
   - Hugging Face上にある `.gguf`
2. `llama.cpp` をビルドする
3. 必要に応じてGGUFへ変換する
4. 必要に応じて量子化する
5. READMEを生成する
6. 生成物をHugging Faceへアップロードする

---

## LM Studioとの関係

このNotebook自体は**LM Studioを操作しません**。  
LM Studioは、**出来上がったGGUFモデルをローカルで実行する側のアプリ**です。

関係性を整理すると次のようになります。

- **このNotebook**: GGUFを作る / 整理する / 配布する
- **Hugging Face**: モデルを保存・共有する
- **LM Studio**: GGUFモデルをローカルで読み込んで推論する

つまり、

**Notebook → Hugging Faceへ保存 → LM Studioで使う**

という流れです。

なお、Hugging FaceへPushしなくても、ローカルにGGUFがあればLM Studioへ直接取り込めます。  
Hugging FaceへPushする主な理由は、**共有・再利用・バージョン管理・複数端末からの利用をしやすくするため**です。

---

## GGUFとは何か

GGUFは、主に `llama.cpp` 系の推論環境で使われる**推論向けモデル形式**です。  
Hugging Faceの通常のTransformers形式（`config.json`、`tokenizer`、`safetensors` など）に比べて、**ローカル推論向けの取り回しがしやすい**のが特徴です。

GGUFを使う利点の例:

- `llama.cpp` 系でそのまま扱いやすい
- 単一ファイルとして配布しやすい
- 量子化済みモデルを扱いやすい
- ローカルPCでの推論に向いている

---

## 量子化とは何か

量子化は、**モデルの重みをより軽量な表現に変換して、サイズやメモリ使用量を減らす処理**です。  
一般に、量子化を行うと次のような効果があります。

- モデルファイルが小さくなる
- 推論時のメモリ使用量が減る
- ローカル環境で動かしやすくなる

一方で、量子化を強くしすぎると、**品質が下がる可能性**があります。  
このNotebookでは、量子化方式を次の候補から選べるようにしています。

- `Q2_K`（最小・最低品質）
- `Q3_K_S`
- `Q3_K_M`
- `Q3_K_L`
- `Q4_K_S`
- `Q4_K_M`
- `Q5_K_S`
- `Q5_K_M`
- `Q6_K`
- `Q8_0`（最大・最高品質）

---

## このNotebookで想定している用途

- Hugging Face上の**マージ済みフルモデル**をGGUFへ変換したい
- 手元のGGUFを整理し、README付きでHugging Faceへ再公開したい
- Hugging Face上の既存GGUFを自分のrepoへ再配置したい
- LM Studioや `llama.cpp` 系ツールで使いやすい形にしたい
- GitHub / Qiitaに公開できる形で処理を明文化したい

---

## 重要な注意点

### 1. LoRAアダプタ単体repoは通常そのまま変換できません
Transformers形式からGGUFへ変換する場合、通常は**マージ済みのフルモデル一式**が必要です。  
LoRA adapterだけしかないrepoは、このNotebookの `hf_transformers` モードの想定外です。

### 2. `llama.cpp` 側の対応が必要です
モデルによっては、`llama.cpp` がそのアーキテクチャを十分にサポートしていないと変換できない場合があります。

### 3. セキュリティのためトークンは表示しません
Hugging Face Token はColabの `userdata` または環境変数から取得しますが、**値を表示しない**実装にしています。

---

## 使い分けの目安

### `SOURCE_MODE = "hf_transformers"`
Hugging Face上の通常のTransformers形式モデルを取得し、**GGUFへ変換して量子化**したい場合。

### `SOURCE_MODE = "local_gguf"`
ローカルにある `.gguf` を入力として使い、**必要に応じて再量子化して再公開**したい場合。

### `SOURCE_MODE = "hf_gguf"`
Hugging Face上の `.gguf` を取得し、**必要に応じて再量子化して自分のrepoへ再公開**したい場合。

---

## 実行前の前提

- Colab などのLinux系環境を想定
- Hugging Face Token を設定済みであること
- 変換元モデル / 入力GGUF へアクセス可能であること
- 変換元がTransformers形式の場合は、**フルモデル一式**であること

In [ ]:
# ============================================================
# 設定セル
# ------------------------------------------------------------
# このセルだけを主に編集すれば、Notebook全体の挙動を切り替えられるようにしています。
# セキュリティ上、トークン値そのものは表示しません。
# ============================================================

from pathlib import Path
import os
import re

# ------------------------------------------------------------
# 入力ソースの種類
#   - "hf_transformers": Hugging Face上のTransformers形式モデルからGGUFへ変換
#   - "local_gguf"     : ローカルのGGUFファイルを入力として使用
#   - "hf_gguf"        : Hugging Face上のGGUFファイルを入力として使用
# ------------------------------------------------------------
SOURCE_MODE = "hf_transformers"

# ------------------------------------------------------------
# 量子化方式
# ユーザー要件に基づき、許可する候補を明示的に制限します。
# 候補外を指定した場合は assert で停止します。
# ------------------------------------------------------------
QUANTIZATION_METHOD = "Q4_K_M"

ALLOWED_QUANTIZATION_METHODS = [
    "Q2_K",
    "Q3_K_S",
    "Q3_K_M",
    "Q3_K_L",
    "Q4_K_S",
    "Q4_K_M",
    "Q5_K_S",
    "Q5_K_M",
    "Q6_K",
    "Q8_0",
]

assert SOURCE_MODE in ["hf_transformers", "local_gguf", "hf_gguf"], (
    f"SOURCE_MODE が不正です: {SOURCE_MODE}"
)
assert QUANTIZATION_METHOD in ALLOWED_QUANTIZATION_METHODS, (
    f"QUANTIZATION_METHOD が不正です: {QUANTIZATION_METHOD}\n"
    f"許可値: {ALLOWED_QUANTIZATION_METHODS}"
)

# ------------------------------------------------------------
# 変換元 / 入力モデルの指定
# SOURCE_MODE に応じて使われる変数が変わります。
# ------------------------------------------------------------

# 1) SOURCE_MODE == "hf_transformers" の場合
#    ここには "Transformers形式のフルモデルrepo" を指定してください。
HF_TRANSFORMERS_MODEL_ID = "your-username/your-merged-model"

# 2) SOURCE_MODE == "local_gguf" の場合
#    ローカルに存在するGGUFファイルのパスを指定してください。
LOCAL_GGUF_PATH = "/content/input_model.gguf"

# 3) SOURCE_MODE == "hf_gguf" の場合
#    Hugging Face上のGGUF repoと、必要なら対象ファイル名を指定してください。
HF_GGUF_REPO_ID = "your-username/your-existing-gguf-repo"
HF_GGUF_FILENAME = ""  # 空文字なら repo 内の最初の .gguf を自動検出

# ------------------------------------------------------------
# 出力先のHugging Face repo
# 最終的なGGUFやREADMEをこのrepoへアップロードします。
# ------------------------------------------------------------
OUTPUT_HF_REPO_ID = "your-username/your-model-gguf"

# ------------------------------------------------------------
# READMEや出力ファイル名に使う表示名
# 空なら自動生成します。
# ------------------------------------------------------------
DISPLAY_MODEL_NAME = ""
README_TITLE = ""
MODEL_DESCRIPTION_JA = (
    "Hugging Face上またはローカルのモデル/GGUFを元に、"
    "GGUF形式へ変換または再量子化し、README付きで再公開したモデルです。"
)

# ------------------------------------------------------------
# 作業ディレクトリ
# ------------------------------------------------------------
WORK_DIR = Path("/content/gguf_work")
LLAMA_CPP_DIR = Path("/content/llama.cpp")
PUBLISH_DIR = WORK_DIR / "publish"
INPUT_DIR = WORK_DIR / "input"
OUTPUT_DIR = WORK_DIR / "output"

# ------------------------------------------------------------
# 中間・出力ファイル名
# 基本的には自動生成しますが、必要なら明示指定できます。
# 空文字の場合は自動命名です。
# ------------------------------------------------------------
F16_OUTPUT_FILENAME = ""        # 例: "my-model-f16.gguf"
QUANTIZED_OUTPUT_FILENAME = ""  # 例: "my-model-Q4_K_M.gguf"

# ------------------------------------------------------------
# README内に記載する補足情報
# ------------------------------------------------------------
BASE_MODEL_NOTE = ""   # 例: "Base model: Qwen/Qwen3-4B-Instruct-2507"
LICENSE_NOTE = ""      # 例: "Please follow the original model license."
USAGE_NOTE = (
    "LM Studio や llama.cpp 系ツールでのローカル推論、"
    "および配布・共有を想定しています。"
)

# ------------------------------------------------------------
# llama.cpp の取得設定
# 原則として main ブランチの最新を使います。
# 特定commitに固定したい場合は GIT_CHECKOUT_REF を指定してください。
# ------------------------------------------------------------
LLAMA_CPP_REPO_URL = "https://github.com/ggml-org/llama.cpp.git"
GIT_CHECKOUT_REF = ""  # 例: "b4837" / "master" / "main" / specific commit hash

# ------------------------------------------------------------
# Hugging Face Token の取得
# Colab の userdata を優先し、なければ環境変数を見ます。
# 値は表示しません。
# ------------------------------------------------------------
HF_TOKEN = None

try:
    from google.colab import userdata  # Colab専用
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass

if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN")

assert HF_TOKEN, (
    "Hugging Face Token が取得できませんでした。"
    "Colab userdata の HF_TOKEN または環境変数 HF_TOKEN を設定してください。"
)

# ------------------------------------------------------------
# ユーティリティ: 表示名やファイル名用の文字列整形
# ------------------------------------------------------------
def sanitize_name(text: str) -> str:
    text = text.strip()
    text = text.replace("/", "-")
    text = re.sub(r"[^0-9A-Za-z._-]+", "-", text)
    text = re.sub(r"-{2,}", "-", text).strip("-")
    return text

def infer_source_label() -> str:
    if SOURCE_MODE == "hf_transformers":
        return HF_TRANSFORMERS_MODEL_ID
    elif SOURCE_MODE == "local_gguf":
        return Path(LOCAL_GGUF_PATH).stem
    elif SOURCE_MODE == "hf_gguf":
        return HF_GGUF_REPO_ID
    raise ValueError(f"Unknown SOURCE_MODE: {SOURCE_MODE}")

SOURCE_LABEL = infer_source_label()
AUTO_MODEL_STEM = sanitize_name(SOURCE_LABEL.split("/")[-1])

if not DISPLAY_MODEL_NAME:
    DISPLAY_MODEL_NAME = AUTO_MODEL_STEM

if not README_TITLE:
    README_TITLE = f"{DISPLAY_MODEL_NAME} (GGUF)"

if not F16_OUTPUT_FILENAME:
    F16_OUTPUT_FILENAME = f"{AUTO_MODEL_STEM}-f16.gguf"

if not QUANTIZED_OUTPUT_FILENAME:
    QUANTIZED_OUTPUT_FILENAME = f"{AUTO_MODEL_STEM}-{QUANTIZATION_METHOD}.gguf"

F16_OUTPUT_PATH = OUTPUT_DIR / F16_OUTPUT_FILENAME
QUANTIZED_OUTPUT_PATH = OUTPUT_DIR / QUANTIZED_OUTPUT_FILENAME

print("設定の検証が完了しました。")
print(f"SOURCE_MODE          : {SOURCE_MODE}")
print(f"QUANTIZATION_METHOD  : {QUANTIZATION_METHOD}")
print(f"OUTPUT_HF_REPO_ID    : {OUTPUT_HF_REPO_ID}")
print(f"DISPLAY_MODEL_NAME   : {DISPLAY_MODEL_NAME}")
print(f"出力ファイル名       : {QUANTIZED_OUTPUT_FILENAME}")

## ここからの流れ

以降のセルでは、次の順番で処理を進めます。

1. 作業ディレクトリを作成
2. `llama.cpp` を取得してビルド
3. 入力モデル / 入力GGUFを取得
4. 必要に応じてGGUFへ変換
5. 必要に応じて量子化
6. READMEを生成
7. Hugging Faceへアップロード

`requirements.txt` のインストールは不要な前提にし、必要最低限の依存だけを `pip` で入れます。

In [ ]:
# ============================================================
# 必要最小限の依存をインストール
# ------------------------------------------------------------
# requirements.txt は不要との前提のため使いません。
# Hugging Faceとのやり取りに必要な最小限のパッケージのみ導入します。
# ============================================================

!pip -q install -U huggingface_hub

In [ ]:
# ============================================================
# llama.cpp を取得してビルド
# ------------------------------------------------------------
# 余分な make LLAMA_CUDA=1 は使わず、CMakeベースで一本化します。
# GPUが使える環境では CUDA を有効化します。
# ============================================================

import os
import shutil
import subprocess
from pathlib import Path

WORK_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PUBLISH_DIR.mkdir(parents=True, exist_ok=True)

if LLAMA_CPP_DIR.exists():
    shutil.rmtree(LLAMA_CPP_DIR)

subprocess.run(["git", "clone", LLAMA_CPP_REPO_URL, str(LLAMA_CPP_DIR)], check=True)

if GIT_CHECKOUT_REF:
    subprocess.run(["git", "-C", str(LLAMA_CPP_DIR), "checkout", GIT_CHECKOUT_REF], check=True)

build_dir = LLAMA_CPP_DIR / "build"
build_dir.mkdir(parents=True, exist_ok=True)

cmake_cmd = [
    "cmake",
    "..",
    "-DGGML_CUDA=ON",
    "-DCMAKE_BUILD_TYPE=Release",
]

# nvcc が見つからない環境もあるため、存在する場合のみ明示します。
nvcc_path = "/usr/local/cuda/bin/nvcc"
if os.path.exists(nvcc_path):
    cmake_cmd.append(f"-DCMAKE_CUDA_COMPILER={nvcc_path}")

subprocess.run(cmake_cmd, cwd=build_dir, check=True)
subprocess.run(
    ["cmake", "--build", ".", "--config", "Release", "-j", str(os.cpu_count() or 2)],
    cwd=build_dir,
    check=True,
)

print("llama.cpp のビルドが完了しました。")

## 入力モデルの取得

`SOURCE_MODE` に応じて取得方法を切り替えます。

- `hf_transformers`  
  Hugging Face上のTransformers形式モデルrepoをダウンロードします。
- `local_gguf`  
  ローカルのGGUFファイルを入力としてコピーします。
- `hf_gguf`  
  Hugging Face上のGGUF repoから対象 `.gguf` を取得します。

In [ ]:
# ============================================================
# 入力を取得
# ------------------------------------------------------------
# SOURCE_MODE ごとに処理を分岐します。
# ============================================================

import shutil
from pathlib import Path
from huggingface_hub import snapshot_download

TRANSFORMERS_LOCAL_DIR = INPUT_DIR / "transformers_model"
INPUT_GGUF_PATH = None

if SOURCE_MODE == "hf_transformers":
    # Transformers形式のフルモデルrepoをローカルへ取得
    snapshot_download(
        repo_id=HF_TRANSFORMERS_MODEL_ID,
        local_dir=str(TRANSFORMERS_LOCAL_DIR),
        token=HF_TOKEN,
    )
    print(f"Transformers形式モデルを取得しました: {HF_TRANSFORMERS_MODEL_ID}")

elif SOURCE_MODE == "local_gguf":
    # ローカルGGUFを作業用ディレクトリへコピー
    src = Path(LOCAL_GGUF_PATH)
    assert src.exists(), f"LOCAL_GGUF_PATH が存在しません: {src}"

    INPUT_GGUF_PATH = INPUT_DIR / src.name
    shutil.copy2(src, INPUT_GGUF_PATH)
    print(f"ローカルGGUFを取得しました: {INPUT_GGUF_PATH}")

elif SOURCE_MODE == "hf_gguf":
    # Hugging Face上のGGUF repoを取得し、対象 .gguf を選ぶ
    hf_gguf_dir = INPUT_DIR / "hf_gguf_repo"
    snapshot_download(
        repo_id=HF_GGUF_REPO_ID,
        local_dir=str(hf_gguf_dir),
        token=HF_TOKEN,
    )

    if HF_GGUF_FILENAME:
        candidate = hf_gguf_dir / HF_GGUF_FILENAME
        assert candidate.exists(), f"指定されたGGUFが見つかりません: {candidate}"
        INPUT_GGUF_PATH = candidate
    else:
        gguf_files = sorted(hf_gguf_dir.rglob("*.gguf"))
        assert gguf_files, f"repo内に .gguf が見つかりません: {HF_GGUF_REPO_ID}"
        INPUT_GGUF_PATH = gguf_files[0]

    print(f"Hugging Face上のGGUFを取得しました: {INPUT_GGUF_PATH}")

else:
    raise ValueError(f"Unknown SOURCE_MODE: {SOURCE_MODE}")

## GGUF変換

`SOURCE_MODE = "hf_transformers"` のときだけ、Transformers形式からGGUFへ変換します。  
`local_gguf` / `hf_gguf` の場合は、すでにGGUFなのでこの段階では変換不要です。

In [ ]:
# ============================================================
# Transformers形式 → f16 GGUF へ変換
# ------------------------------------------------------------
# SOURCE_MODE == "hf_transformers" のときのみ実行します。
# それ以外は既存GGUFをそのまま次工程へ渡します。
# ============================================================

import shutil
import subprocess

if SOURCE_MODE == "hf_transformers":
    convert_script = LLAMA_CPP_DIR / "convert_hf_to_gguf.py"
    assert convert_script.exists(), f"convert_hf_to_gguf.py が見つかりません: {convert_script}"

    subprocess.run(
        [
            "python3",
            str(convert_script),
            str(TRANSFORMERS_LOCAL_DIR),
            "--outfile",
            str(F16_OUTPUT_PATH),
        ],
        check=True,
    )
    GGUF_FOR_QUANTIZATION = F16_OUTPUT_PATH
    print(f"f16 GGUF を作成しました: {GGUF_FOR_QUANTIZATION}")

else:
    # 既存GGUFを中間入力として扱う
    assert INPUT_GGUF_PATH is not None, "INPUT_GGUF_PATH が未設定です。"
    GGUF_FOR_QUANTIZATION = INPUT_GGUF_PATH
    print(f"既存GGUFを量子化入力として使用します: {GGUF_FOR_QUANTIZATION}")

## 量子化

ここでは、指定した量子化方式で最終GGUFを作ります。  
`hf_transformers` の場合は通常 `f16 GGUF → 指定量子化GGUF`、  
`local_gguf` / `hf_gguf` の場合は `既存GGUF → 指定量子化GGUF` という流れになります。

> 量子化元がすでに量子化済みGGUFである場合、再量子化は品質面で不利になる可能性があります。  
> 可能であれば、量子化前または高精度側のGGUFから量子化する方が望ましいです。

In [ ]:
# ============================================================
# 量子化
# ------------------------------------------------------------
# llama.cpp の量子化バイナリを使って、指定方式のGGUFを生成します。
# ============================================================

import subprocess

quantize_candidates = [
    LLAMA_CPP_DIR / "build" / "bin" / "llama-quantize",
    LLAMA_CPP_DIR / "build" / "bin" / "quantize",
]

quantize_bin = None
for p in quantize_candidates:
    if p.exists():
        quantize_bin = p
        break

assert quantize_bin is not None, (
    "量子化バイナリが見つかりませんでした。"
    "llama.cpp のビルド結果を確認してください。"
)

subprocess.run(
    [
        str(quantize_bin),
        str(GGUF_FOR_QUANTIZATION),
        str(QUANTIZED_OUTPUT_PATH),
        QUANTIZATION_METHOD,
    ],
    check=True,
)

assert QUANTIZED_OUTPUT_PATH.exists(), f"量子化後ファイルが作成されていません: {QUANTIZED_OUTPUT_PATH}"
print(f"量子化が完了しました: {QUANTIZED_OUTPUT_PATH}")

In [ ]:
# ============================================================
# 生成物の確認
# ------------------------------------------------------------
# ファイルサイズを確認します。
# ============================================================

!ls -lh "{OUTPUT_DIR}"

## README生成

配布用repoとして見やすくするため、README.md を自動生成します。  
Hugging Face上には、**GGUF本体だけでなくREADMEも一緒にアップロード**します。

READMEには少なくとも次を含めます。

- このrepoが何のモデルか
- どこから来たモデルか
- 量子化方式
- どういう用途を想定しているか
- LM Studioとの関係
- 注意点

In [ ]:
# ============================================================
# README.md を生成
# ------------------------------------------------------------
# GitHub / Hugging Face / Qiita で見ても意味が伝わるよう、
# ある程度説明を持たせたREADMEを自動生成します。
# ============================================================

from textwrap import dedent

def build_source_summary() -> str:
    if SOURCE_MODE == "hf_transformers":
        return f"- Source type: Hugging Face Transformers model\n- Source repo: `{HF_TRANSFORMERS_MODEL_ID}`"
    elif SOURCE_MODE == "local_gguf":
        return f"- Source type: Local GGUF file\n- Source file: `{Path(LOCAL_GGUF_PATH).name}`"
    elif SOURCE_MODE == "hf_gguf":
        filename = HF_GGUF_FILENAME if HF_GGUF_FILENAME else "(auto-detected first .gguf)"
        return f"- Source type: Hugging Face GGUF model\n- Source repo: `{HF_GGUF_REPO_ID}`\n- Source file: `{filename}`"
    raise ValueError(f"Unknown SOURCE_MODE: {SOURCE_MODE}")

readme_text = f"""---
library_name: gguf
tags:
- gguf
- llama-cpp
- quantized
- local-llm
---

# {README_TITLE}

## Overview

{MODEL_DESCRIPTION_JA}

This repository contains a GGUF model prepared for local inference workflows such as LM Studio and llama.cpp.

## Source

{build_source_summary()}

## Quantization

- Final quantization: `{QUANTIZATION_METHOD}`
- Final file: `{QUANTIZED_OUTPUT_FILENAME}`

## Notes

- This repository is intended for **distribution and local inference**, not for training.
- LM Studio is a **consumer of GGUF**, not a conversion tool itself.
- If the source was a Transformers model, conversion was performed via `llama.cpp`.
- If the source was already a GGUF, it was reused and/or requantized.

## Usage

- Download the `.gguf` file from this repository.
- Import it into LM Studio or use it with llama.cpp-compatible tools.

## Additional Notes

{BASE_MODEL_NOTE if BASE_MODEL_NOTE else "- Base model note: not specified"}
{LICENSE_NOTE if LICENSE_NOTE else "- License note: please follow the original upstream license and usage terms"}
- Intended use: {USAGE_NOTE}
"""

readme_path = PUBLISH_DIR / "README.md"
readme_path.write_text(dedent(readme_text).strip() + "\n", encoding="utf-8")

print(readme_path)
print(readme_path.read_text(encoding="utf-8"))

## Hugging Faceへアップロード

最後に、生成したGGUFとREADMEを同じrepoへアップロードします。  
単一ファイルアップロードではなく、**公開用ディレクトリをまとめてアップロード**する構成にしてあります。

In [ ]:
# ============================================================
# 生成物を公開用ディレクトリへ集約し、Hugging Faceへアップロード
# ------------------------------------------------------------
# GGUF本体とREADMEを一緒にアップロードします。
# ============================================================

import shutil
from huggingface_hub import HfApi

# 公開用ディレクトリへGGUFをコピー
publish_gguf_path = PUBLISH_DIR / QUANTIZED_OUTPUT_PATH.name
shutil.copy2(QUANTIZED_OUTPUT_PATH, publish_gguf_path)

api = HfApi()

api.create_repo(
    repo_id=OUTPUT_HF_REPO_ID,
    repo_type="model",
    exist_ok=True,
    token=HF_TOKEN,
)

api.upload_folder(
    folder_path=str(PUBLISH_DIR),
    repo_id=OUTPUT_HF_REPO_ID,
    repo_type="model",
    token=HF_TOKEN,
)

print("アップロードが完了しました。")
print(f"Hugging Face repo: https://huggingface.co/{OUTPUT_HF_REPO_ID}")

## 補足メモ

- `local_gguf` や `hf_gguf` は「すでにGGUFがある」ケース向けです。  
  すでに強く量子化されたGGUFをさらに再量子化すると、品質劣化の可能性があります。
- `hf_transformers` は、通常のHugging FaceモデルからGGUFを作りたいときの基本ルートです。
- LM StudioはこのNotebookの代替ではなく、**このNotebookの成果物を使うアプリ**です。
- うまく変換できない場合は、元モデルがフルモデルか、`llama.cpp` 側が対象アーキテクチャに対応しているかを確認してください。